<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-00-prerequisites/lesson-0.1-python-for-genai/notebooks/exercises-0.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Lesson 0.1: Python for GenAI — Practice Exercises (Enhanced)

**Netsetos GenAI Course** | Module 0

6 exercises covering async, generators, Pydantic, **error handling (NEW)**, decorators, and the **production LLM call pattern (NEW)**.

**Research-enhanced:** Covers the Python skills that 80% of GenAI courses skip — the exact patterns used in production LLM apps.

---

## Exercise 1 (Easy): Async Speedup — 10 Parallel Calls

### 📚 Theory
LLM API calls are I/O-bound — 95% waiting for network. `asyncio.gather()` fires all calls simultaneously. 10 calls × 2s each: sequential = 20s, async = 2s.

### 📋 Task
1. `async def call_llm(prompt)` with `await asyncio.sleep(2)`
2. 10 prompts, measure sequential vs parallel time

In [ ]:
# YOUR CODE HERE
import asyncio, time

async def call_llm(prompt, delay=2.0):
    # TODO
    pass

async def main():
    prompts = [f'Q{i}' for i in range(10)]
    # TODO: sequential, then parallel with gather

await main()

<details><summary>✅ Solution</summary>



In [ ]:
import asyncio, time
async def call_llm(prompt, delay=2.0):
    await asyncio.sleep(delay)
    return f'Response: {prompt}'

async def main():
    prompts = [f'Q{i}' for i in range(10)]
    start = time.time()
    for p in prompts: await call_llm(p)
    seq = time.time()-start
    start = time.time()
    await asyncio.gather(*[call_llm(p) for p in prompts])
    par = time.time()-start
    print(f'Sequential: {seq:.1f}s | Async: {par:.1f}s | Speedup: {seq/par:.1f}x')
await main()

</details>

---

## Exercise 2 (Easy): Streaming Simulator

### 📚 Theory
Generators + `yield` display tokens as they arrive. TTFT drops from 2000ms to 150ms. Every chat app uses this.

### 📋 Task
1. `stream_chars(text, delay=0.05)` — yield chars
2. `stream_words(text, delay=0.1)` — yield words

In [ ]:
# YOUR CODE HERE
import time
def stream_chars(text, delay=0.05):
    pass  # TODO

for c in stream_chars('Hello!'): print(c, end='', flush=True)

<details><summary>✅ Solution</summary>



In [ ]:
import time
def stream_chars(text, delay=0.05):
    for c in text: time.sleep(delay); yield c
def stream_words(text, delay=0.1):
    for w in text.split(): time.sleep(delay); yield w+' '

for c in stream_chars('Hello from Gemini!'): print(c,end='',flush=True)
print()
for w in stream_words('RAG combines retrieval with generation'): print(w,end='',flush=True)

</details>

---

## Exercise 3 (Medium): Pydantic Model Library

### 📚 Theory
Pydantic is the #1 Python library for GenAI. It validates types AND constraints. OpenAI, LangChain, FastAPI all use it internally. When LLM returns `{"rating": "high"}` instead of `{"rating": 8}`, Pydantic catches it.

### 📋 Task
1. ChatMessage: role (user/assistant/system), content (1-10000), optional tokens
2. CourseRecommendation: name, score (0-1), difficulty, hours
3. CodeReview: file, score (1-10), issues, suggestions
4. Test valid + invalid data

In [ ]:
# YOUR CODE HERE
from pydantic import BaseModel, Field
from typing import Optional

# TODO: 3 models + tests

<details><summary>✅ Solution</summary>



In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ChatMessage(BaseModel):
    role: str = Field(..., pattern='^(user|assistant|system)$')
    content: str = Field(..., min_length=1, max_length=10000)
    tokens: Optional[int] = Field(None, ge=0)

class CourseRecommendation(BaseModel):
    course_name: str = Field(..., min_length=3)
    relevance_score: float = Field(..., ge=0.0, le=1.0)
    difficulty: str = Field(..., pattern='^(beginner|intermediate|advanced)$')
    estimated_hours: int = Field(..., ge=1, le=500)
    topics: list[str] = Field(..., min_length=1)

class CodeReview(BaseModel):
    file_name: str
    overall_score: int = Field(..., ge=1, le=10)
    issues: list[str] = Field(default_factory=list)

msg = ChatMessage(role='assistant', content='Hello!', tokens=5)
print(f'Valid: {msg}')
try: ChatMessage(role='hacker', content='test')
except Exception as e: print(f'Caught bad role: {type(e).__name__}')

</details>

---

## Exercise 4 (Medium): Error Handling with Fallback Chain ⭐ NEW

### 📚 Theory
LLM APIs fail: 429 (rate limit) → wait and retry. 500 (server error) → retry. 401 (auth) → FAIL FAST. The production pattern: exponential backoff with jitter + model fallback chain (primary → cheaper → cached).

### 📋 Task
1. Create RateLimitError, ServerError, AuthError exceptions
2. `retry_with_backoff()` — exponential backoff, fail fast on auth
3. `call_with_fallback()` — try 3 models in sequence
4. Test with 40% failure rate

In [ ]:
# YOUR CODE HERE
import time, random

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

# TODO: retry_with_backoff(), call_with_fallback()

<details><summary>✅ Solution</summary>



In [ ]:
import time, random
class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_unreliable(prompt):
    r = random.random()
    if r < 0.3: raise RateLimitError('429')
    if r < 0.4: raise ServerError('500')
    return f'Answer: {prompt}'

def retry_backoff(func, prompt, retries=3):
    for i in range(retries):
        try: return func(prompt)
        except AuthError: print('  Auth fail — STOP'); raise
        except RateLimitError:
            w = 2**i + random.uniform(0,1)
            print(f'  Rate limited, wait {w:.1f}s'); time.sleep(w)
        except ServerError:
            print(f'  Server error, retry {i+1}'); time.sleep(0.5)
    raise Exception('All retries failed')

def call_fallback(prompt):
    for model in ['primary','cheaper','cached']:
        try:
            print(f'  Trying {model}...')
            if model == 'cached': return '[Cached]'
            return retry_backoff(call_unreliable, prompt)
        except: print(f'  {model} failed')
    return 'Unavailable'

print(call_fallback('What is GenAI?'))

</details>

---

## Exercise 5 (Hard): Decorator Stack — @cache + @retry + @timer

### 📚 Theory
Production LLM calls stack decorators: `@timer @retry @cache @func`. Order matters. Cache is checked FIRST (innermost), retry wraps it, timer measures total. A cached call saves ~₹0.08. At 100K calls/day caching 40% = ₹3,200/day.

### 📋 Task
1. `@cache_response` — dict cache
2. `@retry(max_attempts=3, delay=0.5)` — exponential backoff
3. `@timer` — print ms
4. Stack all three, test with 30% failure rate

In [ ]:
# YOUR CODE HERE
import time, functools, random

# TODO: cache_response, retry, timer decorators
# TODO: stack on call_llm, test twice

<details><summary>✅ Solution</summary>



In [ ]:
import time, functools, random
def cache_response(func):
    _c = {}
    @functools.wraps(func)
    def w(*a):
        k=str(a)
        if k in _c: print('  CACHE HIT'); return _c[k]
        r=func(*a); _c[k]=r; return r
    return w

def retry(n=3, delay=0.5):
    def dec(func):
        @functools.wraps(func)
        def w(*a):
            for i in range(n):
                try: return func(*a)
                except:
                    if i<n-1: time.sleep(delay*2**i)
                    else: raise
        return w
    return dec

def timer(func):
    @functools.wraps(func)
    def w(*a):
        s=time.time(); r=func(*a); print(f'  {(time.time()-s)*1000:.0f}ms'); return r
    return w

@timer
@retry(3, 0.3)
@cache_response
def call_llm(p):
    if random.random()<0.3: raise ConnectionError('timeout')
    time.sleep(0.5); return f'Answer: {p}'

print('Call 1:'); print(call_llm('RAG'))
print('Call 2 (cached):'); print(call_llm('RAG'))

</details>

---

## Exercise 6 (Hard): Production LLM Pipeline — ALL Skills ⭐ NEW

### 📚 Theory
This combines ALL 7 skills into the exact architecture used in Modules 5-11: env vars → Pydantic schema → async parallel calls → retry on failure → cache duplicates → validate response → stream output.

### 📋 Task
1. Pydantic: `LLMResponse(answer, confidence, tokens)`
2. Async `@cache` decorator
3. Async `call_llm()` with retry + Pydantic validation
4. Generator `stream()` for word-by-word output
5. Run 4 prompts (including duplicate) in parallel

In [ ]:
# YOUR CODE HERE
import asyncio, os, time, functools, random
from pydantic import BaseModel, Field

# TODO: Full pipeline combining all 7 skills

<details><summary>✅ Solution</summary>



In [ ]:
import asyncio, os, time, functools, random
from pydantic import BaseModel, Field

class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

def cache(func):
    _c = {}
    @functools.wraps(func)
    async def w(*a):
        k=str(a)
        if k in _c: print(f'  CACHE: {a[0]}'); return _c[k]
        r = await func(*a); _c[k]=r; return r
    w.cache = _c; return w

@cache
async def call_llm(prompt):
    for i in range(3):
        try:
            await asyncio.sleep(random.uniform(0.3,0.8))
            if random.random()<0.2: raise ConnectionError('timeout')
            return LLMResponse(answer=f'GenAI for {prompt}',
                confidence=round(random.uniform(0.7,0.99),2), tokens=random.randint(30,150))
        except:
            if i<2: await asyncio.sleep(2**i)
            else: raise

def stream(text, delay=0.02):
    for w in text.split(): time.sleep(delay); yield w+' '

async def main():
    start = time.time()
    results = await asyncio.gather(*[call_llm(p) for p in ['RAG','agents','MCP','RAG']])
    print(f'4 calls in {time.time()-start:.1f}s\n')
    for r in results:
        print(f'[{r.confidence:.0%}, {r.tokens}tok] ', end='')
        for w in stream(r.answer): print(w, end='', flush=True)
        print()

await main()

</details>

---

## 🎉 Done!

You've mastered all 8 Python skills for GenAI:
- **Async/Await** — parallel API calls
- **Generators** — token streaming
- **Type Hints + Pydantic** — LLM output validation
- **Decorators** — retry, cache, timer
- **Env Variables + JSON** — secure keys, clean data
- **Error Handling** — rate limits, fallback chains ⭐ NEW
- **Production Pattern** — all skills combined ⭐ NEW

No other GenAI course teaches this integrated set. Next: Lesson 0.2 Math for GenAI.